# All Required Package import

In [19]:
import cv2
import pandas as pd
import os
import numpy as np
from PIL import Image
import shutil
import random
import sklearn

# Path of image and csv file 

In [20]:
image_folder = r"C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\Assessment 3 in number plate detection\new1\license_plates_detection_train"
csv_path = r"C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\Assessment 3 in number plate detection\new\Licplatesdetection_train.csv"

# Read Csv filem 

In [21]:
plate_info = pd.read_csv(csv_path)
print(plate_info)

      img_id  ymin  xmin  ymax  xmax
0      1.jpg   276    94   326   169
1     10.jpg   311   395   344   444
2    100.jpg   406   263   450   434
3    101.jpg   283   363   315   494
4    102.jpg   139    42   280   222
..       ...   ...   ...   ...   ...
895   95.jpg   426    34   508   140
896   96.jpg   356   378   457   548
897   97.jpg   229   149   283   217
898   98.jpg   272   252   300   383
899   99.jpg    53   503   217   569

[900 rows x 5 columns]


# Creating the Folder 

In [22]:
path=os.getcwd()
resized_images_folder = "resized_images"  # Folder for resized images (512x512)
labels_folder = "labels" # Folder for YOLO annotations

dir = os.path.join(path,resized_images_folder)
dir1 = os.path.join(path,labels_folder)

try:
    os.makedirs(dir, exist_ok=True)
    os.makedirs(dir1, exist_ok=True)
except :
    print('Both folder are already exit ')

# Resize the image and make the annotetion in the form of text in labels folderm 

In [ ]:
# Target size
target_size = 512

# Process each image
for _, row in plate_info.iterrows():
    image_name = row["img_id"]
    image_path = os.path.join(image_folder, image_name)
    
    # Check if the image exists
    if not os.path.exists(image_path):
        print(f"⚠️ Warning: Image {image_name} not found. Skipping...")
        continue

    # Open image
    with Image.open(image_path) as img:
        orig_w, orig_h = img.size  # Original dimensions
        
        # Resize image to 512x512
        img_resized = img.resize((target_size, target_size))
        resized_image_path = os.path.join(dir, image_name)
        img_resized.save(resized_image_path)

    # Get bounding box values from CSV
    x_min, y_min, x_max, y_max = row["xmin"], row["ymin"], row["xmax"], row["ymax"]

    # Adjust bounding box based on new size
    x_min_resized = (x_min / orig_w) * target_size
    y_min_resized = (y_min / orig_h) * target_size
    x_max_resized = (x_max / orig_w) * target_size
    y_max_resized = (y_max / orig_h) * target_size

    # Convert to YOLO format (normalize by 512)
    x_center = (x_min_resized + x_max_resized) / 2 / target_size
    y_center = (y_min_resized + y_max_resized) / 2 / target_size
    width = (x_max_resized - x_min_resized) / target_size
    height = (y_max_resized - y_min_resized) / target_size

    # Define YOLO class ID (e.g., 0 for "number plate")
    class_id = 0

    # Save annotation in YOLO format
    label_path = os.path.join(dir1, image_name.replace(".jpg", ".txt").replace(".png", ".txt"))
    with open(label_path, "w") as f:
        f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

print("Resizing and YOLO conversion complete!")


# Rearrenge the folder structure and creat the folder of image and annotetion into train and test split form

In [10]:
# Define dataset root directory
dataset_path = os.path.join(os.getcwd(), "dataset")  # Creates "dataset" in the current working directory
os.makedirs(dataset_path, exist_ok=True)

# Define images and labels directory inside "dataset"
images_path = os.path.join(dataset_path, "images")
labels_path = os.path.join(dataset_path, "labels")
os.makedirs(images_path, exist_ok=True)
os.makedirs(labels_path, exist_ok=True)

# Define train/val split directories
train_img_dir = os.path.join(images_path, "train")
val_img_dir = os.path.join(images_path, "val")
train_label_dir = os.path.join(labels_path, "train")
val_label_dir = os.path.join(labels_path, "val")

# Create all necessary directories
os.makedirs(train_img_dir, exist_ok=True)
os.makedirs(val_img_dir, exist_ok=True)
os.makedirs(train_label_dir, exist_ok=True)
os.makedirs(val_label_dir, exist_ok=True)

# Print final paths for verification
print("Dataset directories created successfully!")
print(f"Dataset Root: {dataset_path}")
print(f"Train Images: {train_img_dir}")
print(f"Val Images: {val_img_dir}")
print(f"Train Labels: {train_label_dir}")
print(f"Val Labels: {val_label_dir}")


Dataset directories created successfully!
Dataset Root: C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\dataset
Train Images: C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\dataset\images\train
Val Images: C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\dataset\images\val
Train Labels: C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\dataset\labels\train
Val Labels: C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\dataset\labels\val


# Move the training and testing image and anotetion text file into there respective folder

In [11]:
# Original image and label paths (update these if needed)
images_path=os.path.join(path,'resized_images')
labels_path=os.path.join(path,'labels')

# Define train/val split ratio
train_ratio = 0.8  # 80% train, 20% val

# Get all image filenames
image_filenames = [f for f in os.listdir(images_path) if f.endswith(('.jpg', '.png'))]

# Shuffle and split data
random.shuffle(image_filenames)
train_size = int(len(image_filenames) * train_ratio)

train_images = image_filenames[:train_size]
val_images = image_filenames[train_size:]

# Move images and labels to respective folders
for img_name in train_images:
    # Move image
    shutil.move(os.path.join(images_path, img_name), train_img_dir)
    
    # Move corresponding label file
    label_name = img_name.rsplit(".", 1)[0] + ".txt"  # Convert image name to label name
    label_path = os.path.join(labels_path, label_name)
    
    if os.path.exists(label_path):
        shutil.move(label_path, train_label_dir)
    else:
        print(f"⚠️ Warning: No label found for {img_name}, skipping...")

for img_name in val_images:
    # Move image
    shutil.move(os.path.join(images_path, img_name), val_img_dir)
    
    # Move corresponding label file
    label_name = img_name.rsplit(".", 1)[0] + ".txt"  # Convert image name to label name
    label_path = os.path.join(labels_path, label_name)
    
    if os.path.exists(label_path):
        shutil.move(label_path, val_label_dir)
    else:
        print(f"⚠️ Warning: No label found for {img_name}, skipping...")

print("Dataset successfully organized into train/val split!")


Dataset successfully organized into train/val split!


# Check the image and` corresponding text file is splir in train and val

In [14]:
def check_image_label_pairs(images_dir, labels_dir, dataset_name="Dataset"):
    print(f"\nChecking image-label pairs in {dataset_name}...")

    image_files = sorted([f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    label_files = sorted([f for f in os.listdir(labels_dir) if f.endswith('.txt')])

    image_names = set(os.path.splitext(f)[0] for f in image_files)
    label_names = set(os.path.splitext(f)[0] for f in label_files)

    missing_labels = image_names - label_names
    missing_images = label_names - image_names

    if missing_labels:
        print(f"{len(missing_labels)} image(s) with no matching label:")
        for name in missing_labels:
            print(f" - {name}")
    if missing_images:
        print(f"{len(missing_images)} label(s) with no matching image:")
        for name in missing_images:
            print(f" - {name}")
    if not missing_labels and not missing_images:
        print(f"All images and labels match in {dataset_name}.")

# path
path=os.getcwd()
train_images_dir = os.path.join(path,'dataset\\images\\train')
train_labels_dir = os.path.join(path,'dataset\\labels\\train')

test_images_dir = os.path.join(path,'dataset\\images\\val')
test_labels_dir = os.path.join(path,'dataset\\labels\\val')

# === Check both sets ===
check_image_label_pairs(train_images_dir, train_labels_dir, "Training Set")
check_image_label_pairs(test_images_dir, test_labels_dir, "Test Set")



Checking image-label pairs in Training Set...
All images and labels match in Training Set.

Checking image-label pairs in Test Set...
All images and labels match in Test Set.


In [17]:
from ultralytics import YOLO

# Load YOLOv8 model
model = YOLO("yolov8n.pt")

# Train with settings to reduce overfitting
model.train(
    data="dataset/data.yaml",
    epochs=50,
    batch=8,
    imgsz=512,
    device="cpu",
    patience=10,           # Early stopping
    val=True,              # Ensure validation runs
    lr0=0.001,             # Lower starting learning rate
    lrf=0.01,              # Final learning rate factor
    weight_decay=0.001,    # L2 regularization (default 0.0005, increase slightly)
    hsv_h=0.015,           # Small color hue augmentation
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5,             # Small rotation for augmentation
    translate=0.1,         # Slight shifting
    scale=0.5,             # Image scaling
    shear=0.1,             # Minor shearing
    perspective=0.0005,    # Add minor perspective warping
    flipud=0.0,            # No vertical flipping (not useful for plates)
    fliplr=0.5,            # Enable horizontal flipping
    mosaic=1.0,            # Mosaic augmentation
    mixup=0.2              # MixUp augmentation
)


New https://pypi.org/project/ultralytics/8.3.102 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.69  Python-3.12.4 torch-2.5.1+cpu CPU (11th Gen Intel Core(TM) i5-1135G7 2.40GHz)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=dataset/data.yaml, epochs=50, time=None, patience=10, batch=8, imgsz=512, save=True, save_period=-1, cache=False, device=cpu, workers=8, project=None, name=train2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_fr

train: Scanning C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\dataset\labels\train... 72


train: New cache created: C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\dataset\labels\train.cache


val: Scanning C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\dataset\labels\val... 180 im


val: New cache created: C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\dataset\labels\val.cache
Plotting labels to runs\detect\train2\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.001' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.001), 63 bias(decay=0.0)
TensorBoard: model graph visualization added 
Image sizes 512 train, 512 val
Using 0 dataloader workers
Logging results to runs\detect\train2
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

                   all        180        180      0.999      0.989      0.995      0.728



50 epochs completed in 5.436 hours.
Optimizer stripped from runs\detect\train2\weights\last.pt, 6.2MB
Optimizer stripped from runs\detect\train2\weights\best.pt, 6.2MB

Validating runs\detect\train2\weights\best.pt...
Ultralytics 8.3.69  Python-3.12.4 torch-2.5.1+cpu CPU (11th Gen Intel Core(TM) i5-1135G7 2.40GHz)
Model summary (fused): 168 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [00:19


                   all        180        180      0.998      0.989      0.995      0.751
Speed: 2.7ms preprocess, 99.2ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to runs\detect\train2


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001B40DE5EC30>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [18]:
results = model(r"C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\dataset\images\val")  
results.show() 



image 1/180 C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\dataset\images\val\104.jpg: 512x512 1 number plate, 150.4ms
image 2/180 C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\dataset\images\val\105.jpg: 512x512 1 number plate, 95.5ms
image 3/180 C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\dataset\images\val\106.jpg: 512x512 1 number plate, 105.9ms
image 4/180 C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\dataset\images\val\107.jpg: 512x512 1 number plate, 123.3ms
image 5/180 C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\dataset\images\val\108.jpg: 512x512 1 number plate, 125.9ms
image 6/180 C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\dataset\images\val\112.jpg: 512x512 1 number plate, 143.2ms
image 7/180 C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\dataset\images\va

AttributeError: 'list' object has no attribute 'show'

In [ ]:
# Load your trained YOLOv8 model
model = YOLO(r"C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\newplate\runs\detect\train2\weights\best.pt")  # replace with your path

# Load video
video_path = r"C:\Users\mohit\Desktop\Naresh_IT\Full_Stack_Data_Science\INTERNSHIP\License_plate\demo.mp4"   
cap = cv2.VideoCapture(video_path)

# Get video writer setup
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("output_video.mp4", fourcc, cap.get(cv2.CAP_PROP_FPS),
                      (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Inference with YOLO
    results = model(frame)

    # Plot results on frame
    annotated_frame = results[0].plot()

    # Write frame to output video
    out.write(annotated_frame)

    # (Optional) Display live preview
    cv2.imshow("YOLOv8 Detection", annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup
cap.release()
out.release()
cv2.destroyAllWindows()
